# Object Detection: Cat and Dog Breed Classification

This notebook implements an object detection model to classify and locate cats and dogs, as well as identify their breeds, in images.

It utilizes the Oxford-IIIT Pet Dataset and a Single Shot MultiBox Detector (SSD) model for this purpose.

## Import Libraries

In [ ]:
# Data Manipulation and Numerical Libraries
import numpy as np
import pandas as pd
import random

# Deep Learning Libraries (PyTorch)
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import torchvision.transforms.v2.functional as F
from torchvision.models import resnet34
from torchvision.models.detection import ssd
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator

# Data Preprocessing and Utilities
from sklearn.model_selection import train_test_split
from collections import OrderedDict
from sklearn.metrics import average_precision_score
from tqdm import tqdm

# Image Processing and Visualization
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# File I/O and XML Parsing
import os
import xml.etree.ElementTree as ET
import kagglehub

# Weight computing
from sklearn.utils.class_weight import compute_class_weight

## Data Analysis

This section focuses on downloading, loading, and exploring the dataset.

It includes examining the dataset structure, file organization, and basic statistics.

In [ ]:
# Download the dataset from Kaggle using kagglehub
path = kagglehub.dataset_download("zippyz/cats-and-dogs-breeds-classification-oxford-dataset")

print(f"Dataset path: {path}")

GPU setting

In [ ]:
# Determine if a GPU is available and set the device accordingly
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

File Path

In [ ]:
# Define file paths for train/validation, test, images, and XML annotations
trainval_file_path = os.path.join(path, "annotations", "annotations", "trainval.txt")
test_file_path = os.path.join(path, "annotations", "annotations", "test.txt")

image_dir = os.path.join(path, "images", "images")
xml_dir = os.path.join(path, "annotations", "annotations", "xmls")

# Load train/validation data and set column names
df_trainval = pd.read_csv(trainval_file_path, sep="\s+", header=None)
df_trainval.columns = ["Image", "ClassID", "Species", "BreedID"]

# Load test data and set column names
df_test = pd.read_csv(test_file_path, sep="\s+", header=None)
df_test.columns = ["Image", "ClassID", "Species", "BreedID"]

# Check the size of the train/validation and test datasets
print(f"Train/Validation data count: {len(df_trainval)}")
print(f"Test data count: {len(df_test)}")

# Count the number of XML files in the annotation directory
xml_files = [file for file in os.listdir(xml_dir) if file.endswith(".xml")]
print(f"XML files count: {len(xml_files)}")

## Dataset Structure and Imbalance Analysis

This section analyzes the structure of the dataset and addresses the potential issue of class imbalance.

**Assumption**: The train and validation data are combined in the provided `trainval.txt` file. Splitting this data will result in a smaller training dataset compared to the test dataset. I must therefore take into account the purpose of the validation set.

-   **Validation Purpose**: The validation set will be used for early stopping and hyperparameter tuning.

In [ ]:
print(df_trainval.shape)
df_trainval.tail()

In [ ]:
print(df_test.shape)
df_test.tail()

In [ ]:
df_trainval['Species'].value_counts()

In [ ]:
df_test['Species'].value_counts()

Addressing Class Imbalance

 **Observation**: Class imbalance is a potential issue in this dataset and may need to be addressed to improve model performance.

 - **Strategy**: Since the validation set is primarily used for hyperparameter tuning and early stopping, we can prioritize a larger training dataset by reducing the size of the validation split.
 - **Conditional Image Augmentation**:  Apply stronger augmentation techniques to the underrepresented classes to artificially increase their presence in the training data.
 - **Class Weighting**: Class weights can be adapted for object detection to address class imbalance. Object detection models typically have two loss components:
     - **Classification Loss**: This component predicts the class label of each detected object. Class weighting can be applied here.
     - **Localization Loss**: This component predicts the bounding box coordinates.

**Hypothesis**: Initially, conditional augmentation will be the primary strategy. If it does not effectively mitigate the class imbalance, then class weighting will be considered.

## Image Visualization

This section displays example images from the dataset to provide a visual understanding of the data.

In [ ]:
trainval_taglist = df_trainval['Image'].tolist()
test_taglist = df_test["Image"].tolist()

In [ ]:
# Load and display an example image from the train/validation set
example_img_tag = trainval_taglist[0]
example_img_path = os.path.join(image_dir, f"{example_img_tag}.jpg")

# Load the image using OpenCV and convert from BGR to RGB
ex_img = cv2.imread(example_img_path)
ex_img = cv2.cvtColor(ex_img, cv2.COLOR_BGR2RGB)

# Display the example image
plt.imshow(ex_img)
plt.title(f"Example Image: {example_img_tag}")
plt.axis("off")
plt.show()

In [ ]:
# Set example image file and tag from the test set
example_img_tag = test_taglist[0]
example_img_path = os.path.join(image_dir, f"{example_img_tag}.jpg")

# Load the example image
ex_img = cv2.imread(example_img_path)
ex_img = cv2.cvtColor(ex_img, cv2.COLOR_BGR2RGB)

# Visualize the image
plt.imshow(ex_img)
plt.title(f"Example Image: {example_img_tag}")
plt.axis("off")
plt.show()

The image data is loaded in BGR format. Although it was converted to RGB for visualization, it's important to note that any further processing should be done in RGB. This is especially important for the transform step when creating the dataset, which should be performed in RGB.

## Annotation Loading and Preprocessing

This section focuses on loading and parsing the XML annotation files, which contain the bounding box coordinates and class labels for each object in the images.

In [ ]:
# Create a list of XML file names (excluding the .xml extension)
xml_list = [os.path.splitext(file)[0] for file in os.listdir(xml_dir) if file.endswith(".xml")]

# Check for images in the train/validation list that do not have corresponding XML files
missing_xml = [image for image in trainval_taglist if image not in xml_list]
print(f"Missing XML files for the following images: {len(missing_xml)}")
print(missing_xml)

# Filter the train/validation image list to only include images that have corresponding XML files
trainval_taglist = [image for image in trainval_taglist if image in xml_list]
print(f"Updated train/validation list length: {len(trainval_taglist)}")

### Annotation Analysis

This layer will load and parse the xml files.

In [ ]:
# Specify the path for the example XML file (using the first file in the list)
example_xml_file = os.path.join(xml_dir, xml_files[0])

# Parse the XML file
tree = ET.parse(example_xml_file)
root = tree.getroot()

# Recursive function to print all tags and data within an XML element, useful for understanding the XML structure
def print_all_elements(element, indent=""):
    print(f"{indent}{element.tag}: {element.text}")
    for child in element:
        print_all_elements(child, indent + "  ") # Indent the sub-elements for better readability

print_all_elements(root)

In [ ]:
# Example: Iterate through the XML file and display the class names and bounding box coordinates for each object
for obj in root.findall("object"):
    class_name = obj.find("name").text  # Extract class name
    bndbox = obj.find("bndbox")  # Extract bounding box information
    x_min = int(bndbox.find("xmin").text)  # Extract x_min
    y_min = int(bndbox.find("ymin").text)  # Extract y_min
    x_max = int(bndbox.find("xmax").text)  # Extract x_max
    y_max = int(bndbox.find("ymax").text)  # Extract y_max

    print(f"Class: {class_name}, Bounding Box: ({x_min}, {y_min}, {x_max}, {y_max})")

#### Annotation Preprocessing

This layer will preprocess the information in the xml files.

In [ ]:
annotations = []

# Iterate through each annotation file
for xml_file in xml_files:
    xml_path = os.path.join(xml_dir, xml_file)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    image_name = root.find("filename").text # image file name

    # Iterate through each object in the annotation
    for obj in root.findall("object"):
        class_name = obj.find("name").text # class name
        bndbox = obj.find("bndbox") # bounding box target info
        x_min = int(bndbox.find("xmin").text)
        y_min = int(bndbox.find("ymin").text)
        x_max = int(bndbox.find("xmax").text)
        y_max = int(bndbox.find("ymax").text)

        # save each annotation in dictionary and append it to the list
        annotations.append({
            "image": image_name,
            'class': class_name,
            "bbox" : [x_min, y_min, x_max, y_max]
        })
print(annotations[:5])

### Bounding Box Visualization

This section shows an example image with its bounding boxes drawn on it.

In [ ]:
# Load and visualize an example image with bounding box annotations
example_img_tag = trainval_taglist[0]
example_img_path = os.path.join(image_dir, f"{example_img_tag}.jpg")

# Load the example image using OpenCV and convert from BGR to RGB
ex_img = cv2.imread(example_img_path)
ex_img = cv2.cvtColor(ex_img, cv2.COLOR_BGR2RGB)

# Extract the annotations for the current example image
annots = [anno for anno in annotations if anno["image"] == f"{example_img_tag}.jpg"]

print(annots)

# Draw the bounding boxes and class names on the image
for anno in annots:
    x_min, y_min, x_max, y_max = anno["bbox"]
    cv2.rectangle(
        img=ex_img,
        pt1=(x_min, y_min),  # Top-left corner
        pt2=(x_max, y_max),  # Bottom-right corner
        color=(255, 0, 0),  # Color (red in RGB)
        thickness=2,
    )  # Draw a red rectangle
    cv2.putText(
        img=ex_img,
        text=anno["class"],
        org=(x_min, y_min - 5),  # Place text slightly above the top-left corner
        fontFace=cv2.FONT_HERSHEY_SIMPLEX,
        fontScale=0.5,
        color=(255, 0, 0),  # Text color (red in RGB)
        thickness=2,
    )  # (image, annotation, bottom-left corner position of the text, font type, font size, color (RGB), thickness)

# Visualize the image with bounding boxes
plt.imshow(ex_img)
plt.title(f"Example Image: {example_img_tag}")
plt.axis("off")
plt.show()

## Dataset Class and Data Augmentation

This section defines the custom dataset classes for handling the image and annotation data, along with the implementation of conditional data augmentation.

In [ ]:
class VOCDataset(Dataset):
    def __init__(self, image_dir, annotation_dir, classes, image_list, df_input=None, transforms=None):
        self.image_dir = image_dir  # Directory containing the images
        self.annotation_dir = annotation_dir  # Directory containing the XML annotations
        self.classes = classes  # List of class names
        self.transforms = transforms  # Optional image transforms
        self.image_files = image_list  # List of image file names (without extension)
        self.df_input = df_input

    def __len__(self):
        return len(self.image_files)  # Returns the number of images

    def __getitem__(self, idx):
        # Construct the file paths for the current image and its XML annotation
        image_file = self.image_files[idx] + ".jpg"
        annotation_file = self.image_files[idx] + ".xml"
        image_path = os.path.join(self.image_dir, image_file)
        annotation_path = os.path.join(self.annotation_dir, annotation_file)

        # Load the image using PIL and convert it to RGB
        image = Image.open(image_path).convert("RGB")

        # Load and parse the XML annotation file
        boxes = []
        labels = []
        tree = ET.parse(annotation_path)  # Parse the XML annotation file
        root = tree.getroot()  # Get the root element of the XML file

        if self.df_input is not None:
            species = self.df_input[self.df_input["Image"] == self.image_files[idx]]['Species'].iloc[0]
        else:
            species = None

        # Iterate through the 'object' elements in the XML file
        for obj in root.findall("object"):
            class_name = obj.find("name").text  # Extract the class name
            if class_name not in self.classes:
                continue  # Skip if the class name is not in the list of valid classes
            labels.append(self.classes.index(class_name))  # Append the class index to the labels list

            bndbox = obj.find("bndbox")  # Extract the bounding box information
            x_min = int(bndbox.find("xmin").text)  # Extract x_min
            y_min = int(bndbox.find("ymin").text)  # Extract y_min
            x_max = int(bndbox.find("xmax").text)  # Extract x_max
            y_max = int(bndbox.find("ymax").text)  # Extract y_max
            boxes.append([x_min, y_min, x_max, y_max])  # Append the bounding box coordinates

        # Convert bounding box coordinates and labels to PyTorch tensors
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        # Create a target dictionary containing the bounding boxes and labels
        target = {"boxes": boxes, "labels": labels}

        # Apply transforms if specified
        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target  # Return the image and the target dictionary

In [ ]:
class ConditionalAugmentation:
    def __init__(self, species_ag, p=0.5):
        """
        Conditionally applies a horizontal flip to the image and updates bounding boxes if a specific species is present.

        Args:
            species_ag (int): The species type used to conditionally apply augmentation (e.g., 1 for a certain species).
            p (float): The probability of applying the augmentation if the conditional species is present.
        """
        self.species_ag = species_ag  # Store the species value for conditional augmentation
        self.p = p  # Probability of applying augmentation

    def __call__(self, img: torch.Tensor, target: dict[str, torch.Tensor]):
        """
        Applies a horizontal flip to the image and updates the corresponding bounding boxes if the condition is met.

        Args:
            img (torch.Tensor): The input image tensor.
            target (dict): A dictionary containing the bounding boxes ('boxes') and labels ('labels') tensors.

        Returns:
            tuple: A tuple containing the transformed image tensor and the updated target dictionary.
        """
        if any(label == self.species_ag for label in target["labels"]):  # Check if any label matches the species
            if random.random() < self.p:  # Apply augmentation with probability p
                img = F.hflip(img)  # Horizontally flip the image
                # Update bounding boxes to reflect the flip
                img_width = img.shape[-1]  # Get the width of the image
                target_boxes = target["boxes"]

                # Ensure target_boxes is 2D
                if target_boxes.dim() == 1:
                    target_boxes = target_boxes.unsqueeze(0)  # Add a dimension if it's a single box (1D)

                # Update the x-coordinates of the bounding boxes
                x_min = target_boxes[:, 0].clone()  # Create a copy of the x_min values
                x_max = target_boxes[:, 2].clone()  # Create a copy of the x_max values
                target_boxes[:, 0] = img_width - x_max  # Update x_min: new_x_min = image_width - old_x_max
                target_boxes[:, 2] = img_width - x_min  # Update x_max: new_x_max = image_width - old_x_min

                target["boxes"] = target_boxes  # Update the bounding boxes in the target

        return img, target  # Return the updated image and target

In [ ]:
class TestDataset(Dataset):
    def __init__(self, image_dir, image_list, transforms=None):
        """
        Dataset class for test data, loading images and applying transforms.

        Args:
            image_dir (str): Directory containing the test images.
            image_list (list): List of image file names (without .jpg extension).
            transforms (callable, optional): Image transformations to be applied.
        """
        self.image_dir = image_dir  # Directory of the images
        self.image_files = image_list  # list of file names (without .jpg)
        self.transforms = transforms  # Image transformations

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # image file path
        image_file = self.image_files[idx] + ".jpg" # indexing from the list and add .jpg
        image_path = os.path.join(self.image_dir, image_file)

        # Load Image
        image = Image.open(image_path).convert("RGB")

        # Transform
        if self.transforms:
            image = self.transforms(image)

        return image, self.image_files[idx] # image and file name return

In [ ]:
transform = {
    "train": v2.Compose([
        v2.ToImage(),
        v2.RandomApply([ConditionalAugmentation(species_ag=1, p=0.7)], p=1),  # Apply conditional augmentation
        v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Adjust color properties
        v2.RandomPhotometricDistort(),  # Apply random photometric distortion
        v2.ToDtype(dtype=torch.float32, scale=True)  # Convert to float32 and scale
    ]),

    "val": v2.Compose([
        v2.ToImage(),  # Convert to image
        v2.ToDtype(dtype=torch.float32, scale=True)  # Convert to float32 and scale
        ]),

    "test": v2.Compose([
        v2.ToImage(),  # Convert to image
        v2.ToDtype(dtype=torch.float32, scale=True)  # Convert to float32 and scale
    ])
    }

In [ ]:
# Split the train/validation list into training and validation sets
train_list, val_list = train_test_split(trainval_taglist, test_size=0.3, random_state=42)

# Display the sizes of the training, validation, and test datasets
print(f"Train count; {len(train_list)}")
print(f"val count; {len(val_list)}")
print(f"test count; {len(test_taglist)}")

In [ ]:
# Define the class labels (including background)
classes = ["background", "dog", "cat"]  # background added

# Create the training dataset
train_dataset = VOCDataset(
    image_dir=image_dir,
    annotation_dir=xml_dir,
    classes=classes,
    image_list=train_list,  # train_list
    df_input=df_trainval,
    transforms=transform["train"]
)

# Create the validation dataset
val_dataset = VOCDataset(
    image_dir=image_dir,
    annotation_dir=xml_dir,
    classes=classes,
    image_list=val_list,  # val_list
    transforms=transform["val"]
)

# Create the test dataset
test_dataset = TestDataset(
    image_dir=image_dir,
    image_list=test_taglist,  # test_list
    transforms=transform["test"]
)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Display the sizes of the datasets
print(f"Train Dataset Size: {len(train_dataset)}")
print(f"Validation Dataset Size: {len(val_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")

In [ ]:
image, label = next(iter(train_loader))
print(label)

In [ ]:
# Visualize example images from the training dataset
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
fig2, ax2 = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    image, target = train_dataset[i]

    boxes = target["boxes"]

    image_name = train_dataset.image_files[i]

    img_np = (image.permute(1, 2, 0).numpy() * 255).astype("uint8")  # Multiply by 255 to scale from [0, 1] to [0, 255]
    img_copy = img_np.copy()  # Create a copy so that the original tensor is not affected by cv2.rectangle
    for j, box in enumerate(boxes):
        x_min, y_min, x_max, y_max = map(int, box.tolist())

        cv2.rectangle(
            img=img_copy,
            pt1=(x_min, y_min),  # Top-left corner
            pt2=(x_max, y_max),  # Bottom-Right corner
            color=(255, 0, 0),  # Color (RGB)
            thickness=2
        )  # red

        cv2.putText(
            img=img_copy,
            text=train_dataset.classes[train_dataset[i][1]["labels"].item()],
            org=(x_min, y_min - 15),
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=1.5,
            color=(255, 0, 0),
            thickness=2
        )
    ax.flat[i].imshow(img_copy)
    ax.flat[i].axis("off")
    ax.flat[i].set_title(image_name)

    # Load and visualize the original image for comparison
    if image_name in trainval_taglist:
        index = trainval_taglist.index(image_name)
        example_img_tag = trainval_taglist[index]
        example_img_path = os.path.join(image_dir, f"{example_img_tag}.jpg")

        # Load
        ex_img = cv2.imread(example_img_path)
        ex_img = cv2.cvtColor(ex_img, cv2.COLOR_BGR2RGB)

        annots = [anno for anno in annotations if anno["image"] == f"{image_name}.jpg"]

        for anno in annots:
            x_min, y_min, x_max, y_max = anno["bbox"]
            cv2.rectangle(
                img=ex_img,
                pt1=(x_min, y_min),  # Top-left corner
                pt2=(x_max, y_max),  # Bottom-Right corner
                color=(255, 0, 0),  # Color (RGB)
                thickness=2
            )  # red box
            cv2.putText(
                img=ex_img,
                text=anno["class"],
                org=(x_min, y_min - 15),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.5,
                color=(255, 0, 0),
                thickness=2
            )

        # Image visualize
        ax2.flat[i].imshow(ex_img)
        ax2.flat[i].set_title(example_img_tag)
        ax2.flat[i].axis("off")

fig.suptitle("Augmented")  # use suptitle() for Figure object
fig2.suptitle("Original")

plt.show()

Careful application of positional augmentation is needed, as bounding boxes are defined by the original data's dimensions. The stochastic horizontal flip is performed, and then the bounding box coordinates are updated to match the flipped image. After these steps, further minor image augmentations can be added.

# Model Build (SSD)

In [ ]:
class SSDBackbone(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        # The initial stage of ResNet34 has conv1, bn1, and relu separately; combine them into a sequential block
        layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # conv1 = 7x7 conv, stride=2 (downsample) -> maxpool excluded
        # Layers are saved as is for feature extraction
        layer1 = backbone.layer1  # same size
        layer2 = backbone.layer2  # downsampled
        layer3 = backbone.layer3  # same size
        layer4 = backbone.layer4  # downsampled

        # Combine all feature extractors into a sequential module
        self.features = nn.Sequential(layer0, layer1, layer2, layer3, layer4)

        # Upsampling module: change the output channels and apply the activation function (ReLU)
        self.upsampling = nn.Sequential(
            nn.ConvTranspose2d(in_channels=512, out_channels=512, kernel_size=2, stride=2), # upsample
            nn.ReLU(inplace=True)
        )

        # Extra modules: SSD uses multiple feature map sizes, so extra layers are required
        # Gradually downsample the channel size by adding extra layers in each sequential block in ModuleList
        self.extra = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=1),
                    nn.ReLU(inplace=True)
                ),
                nn.Sequential(
                    nn.Conv2d(1024, 256, kernel_size=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),  # kernel size = 3, feature_map size reduced; image size reduced by stride = 2
                    nn.ReLU(inplace=True)
                ),
                nn.Sequential(
                    nn.Conv2d(512, 128, kernel_size=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # kernel size = 3, feature_map size reduced; image size reduced by stride = 2
                    nn.ReLU(inplace=True)
                ),
                nn.Sequential(
                    nn.Conv2d(256, 128, kernel_size=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(128, 256, kernel_size=3),  # kernel size = 3, feature_map size reduced
                    nn.ReLU(inplace=True),
                ),
                nn.Sequential(
                    nn.Conv2d(256, 128, kernel_size=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(128, 256, kernel_size=3),  # kernel size = 3, feature_map size reduced
                    nn.ReLU(inplace=True),
                ),
                nn.Sequential(
                    nn.Conv2d(256, 128, kernel_size=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(128, 256, kernel_size=4),  # kernel size = 4, feature_map size reduced
                    nn.ReLU(inplace=True),
                )
            ]
        )

    def forward(self, x):
        # Feature extraction from the ResNet model
        x = self.features(x)
        # The first feature map is created by applying the upsampling module
        output = [self.upsampling(x)]

        # Sequentially apply extra modules to create additional feature maps
        for block in self.extra:
            x = block(x)
            output.append(x)

        # Return an OrderedDict (indices are provided for each feature map)
        return OrderedDict([str(i), v] for i, v in enumerate(output))

In [ ]:
# Load the ResNet34 model as the base backbone
backbone = resnet34(weights="DEFAULT")
# Wrap the backbone with the SSDBackbone class module
backbone = SSDBackbone(backbone)
# DefaultBoxGenerator: Method for generating the Anchor box
anchor_generator = DefaultBoxGenerator(
    aspect_ratios=[[0.5, 1 ,2], [0.33, 0.5, 1, 2, 3], [0.33, 0.5, 1, 2, 3], [0.33, 0.5, 1, 2, 3], [0.33, 0.5, 1, 2, 3], [0.5, 1, 2], [0.5, 1, 2]],  # 7 feature maps, 7 anchor boxes
    scales=[0.07, 0.15, 0.33, 0.51, 0.69, 0.87, 1.05, 1.20],
    steps=[8, 16, 32, 64, 100, 300, 512], # cumulative stride
)

# Instantiate the SSD model with the specified backbone, anchor generator, input size, and number of classes
model = ssd.SSD(
    backbone=backbone,
    anchor_generator=anchor_generator,
    size=(512, 512),
    num_classes=len(classes)  # Background, Cat, Dog
).to(device)

### IoU, AP functions

In [ ]:
def calculate_iou(box, boxes):
    """
    Calculate Intersection over Union (IoU) between a box and multiple boxes.

    Args:
        box (array): Single bounding box [x_min, y_min, x_max, y_max].
        boxes (array): Array of bounding boxes [[x_min, y_min, x_max, y_max], ...].

    Returns:
        array: IoU scores for each box in `boxes`.
    """
    # Calculate the coordinates of the intersection rectangle
    x_min = np.maximum(box[0], boxes[:, 0])  # Get the maximum of the minimum x values
    y_min = np.maximum(box[1], boxes[:, 1])  # Get the maximum of the minimum y values
    x_max = np.minimum(box[2], boxes[:, 2])  # Get the minimum of the maximum x values
    y_max = np.minimum(box[3], boxes[:, 3])  # Get the minimum of the maximum y values

    # Calculate the area of intersection
    intersection = np.maximum(0, x_max - x_min) * np.maximum(0, y_max - y_min)  # If there's no intersection, it's 0

    # Calculate the area of each box
    box_area = (box[2] - box[0]) * (box[3] - box[1])  # (xmax - xmin) * (ymax - ymin)
    boxes_area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

    # Calculate the area of union
    union = box_area + boxes_area - intersection

    # Calculate IoU
    iou = intersection / union

    return iou


def calculate_ap(predictions, ground_truths, class_idx, iou_threshold=0.5):
    """
    Calculate Average Precision (AP) for a specific class.

    This function calculates the AP for a given class by comparing the model's
    predictions against the ground truth data. It computes true positives and
    false positives based on the Intersection over Union (IoU) of the predicted
    and ground truth bounding boxes.

    Args:
        predictions (list): Model's predictions, each containing a dictionary with
                            'boxes' (list of bounding boxes) and 'labels' (list of labels).
                            Format: [{"boxes": [[x_min, y_min, x_max, y_max], ...], "labels": [label, ...]}]
        ground_truths (list): Target ground truth data, each containing a dictionary with
                             'boxes' (list of bounding boxes) and 'labels' (list of labels).
                             Format: [{"boxes": [[x_min, y_min, x_max, y_max], ...], "labels": [label, ...]}]
        class_idx (int): The index of the class for which to calculate AP.
        iou_threshold (float): The minimum IoU threshold to consider a prediction a true positive.

    Returns:
        float: The Average Precision (AP) for the specified class.
    """
    true_positives = []  # List to store whether each prediction is a true positive (1) or not (0)
    false_positives = []  # List to store whether each prediction is a false positive (1) or not (0)
    all_ground_truths = 0  # Total number of ground truth boxes for the specified class

    # Iterate through each prediction-ground truth pair
    for pred, gt in zip(predictions, ground_truths):
        pred_boxes = np.array(pred["boxes"])  # Predicted bounding boxes
        pred_labels = np.array(pred["labels"])  # Predicted labels
        gt_boxes = np.array(gt["boxes"])  # Ground truth bounding boxes
        gt_labels = np.array(gt["labels"])  # Ground truth labels

        # Filter predictions and ground truths to only include the specified class
        pred_boxes = pred_boxes[pred_labels == class_idx]
        gt_boxes = gt_boxes[gt_labels == class_idx]

        all_ground_truths += len(gt_boxes)  # Update the total number of ground truths

        # Keep track of ground truths that have been detected
        detected = []
        for pred_box in pred_boxes:
            ious = []  # List to store the IoUs for the current predicted box with all ground truth boxes
            for gt_box in gt_boxes:
                iou = calculate_iou(pred_box, gt_box)  # Calculate IoU
                ious.append(iou)

            if len(ious) > 0:
                max_iou_idx = np.argmax(ious)  # Index of the ground truth box with the highest IoU
                # Check if the highest IoU exceeds the threshold and if this ground truth hasn't been detected before
                if ious[max_iou_idx] >= iou_threshold and max_iou_idx not in detected:
                    true_positives.append(1)  # Mark as true positive
                    false_positives.append(0)
                    detected.append(max_iou_idx)  # Mark ground truth as detected
                else:
                    true_positives.append(0)  # Mark as false positive
                    false_positives.append(1)
            # No IoU with ground truth means this is a false positive
            else:
                false_positives.append(1)

    # Precision-Recall Curve Calculation
    tp_cumsum = np.cumsum(true_positives)  # Cumulative sum of true positives
    fp_cumsum = np.cumsum(false_positives)  # Cumulative sum of false positives
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)  # Precision at each point
    recalls = tp_cumsum / (all_ground_truths + 1e-6)  # Recall at each point

    # AP Calculation
    ap = 0.0
    # Iterate through precision-recall curve points
    for i in range(1, len(precisions)):
        ap += (recalls[i] - recalls[i - 1]) * precisions[i]  # Calculate area under the curve

    return ap

In [ ]:
def evaluate_model(predictions, ground_truths, classes):
    """
    Evaluates the model's performance by calculating the mean Average Precision (mAP)
    across different classes.

    This function calculates the mAP by iterating through each class, determining
    true positives, and calculating the Average Precision (AP) for each class.
    It then averages the APs across all classes to get the mAP.

    Args:
        predictions (list): List of model predictions, where each prediction is a dictionary
                            containing 'boxes', 'labels', and 'scores'.
        ground_truths (list): List of ground truth annotations, where each annotation is a
                             dictionary containing 'boxes' and 'labels'.
        classes (list): List of class names.

    Returns:
        float: The mean Average Precision (mAP) across all classes.
    """
    class_aps = []  # Store AP for each class

    for class_idx, class_name in enumerate(classes[1:], start=1):  # Iterate through each class (excluding background)
        true_positives = []  # Store the true positives for the current class
        scores = []  # Store the confidence scores for the current class
        num_ground_truths = 0  # Count the number of ground truth instances for the current class

        for pred, gt in zip(predictions, ground_truths):  # Iterate through each prediction-ground truth pair
            # Filter for the current class
            pred_boxes = pred["boxes"][pred["labels"] == class_idx].cpu().numpy() if len(pred["boxes"]) > 0 else []
            pred_scores = pred["scores"][pred["labels"] == class_idx].cpu().numpy() if len(pred["scores"]) > 0 else []
            gt_boxes = gt["boxes"][gt["labels"] == class_idx].cpu().numpy() if len(gt["boxes"]) > 0 else []

            num_ground_truths += len(gt_boxes)  # Increment the number of ground truth boxes

            if len(pred_boxes) == 0 or len(gt_boxes) == 0:
                continue  # Skip if no predictions or ground truths for this class

            matched = np.zeros(len(gt_boxes), dtype=bool)  # Keep track of which ground truth boxes have been matched
            for box, score in zip(pred_boxes, pred_scores):  # Iterate through each predicted box and score
                ious = calculate_iou(box, gt_boxes)  # Calculate IoU with each ground truth box
                max_iou_idx = np.argmax(ious) if len(ious) > 0 else -1  # Find the index of the max IoU
                max_iou = ious[max_iou_idx] if max_iou_idx >= 0 else 0  # Get the maximum IoU

                # If the maximum IoU is above the threshold and it's not already matched, count as a true positive
                if max_iou >= 0.5 and not matched[max_iou_idx]:
                    true_positives.append(1)
                    matched[max_iou_idx] = True  # Mark this ground truth box as matched
                else:
                    true_positives.append(0)  # Otherwise, not a true positive

                scores.append(score)  # Store the confidence score

        if len(scores) == 0:
            class_aps.append(0)  # If no scores were found, add 0 as the ap
            continue

        # Sort predictions by confidence score in descending order
        sorted_indices = np.argsort(-np.array(scores))
        true_positives = np.array(true_positives)[sorted_indices]
        scores = np.array(scores)[sorted_indices]

        # Calculate precision and recall
        cum_true_positives = np.cumsum(true_positives)
        precision = cum_true_positives / (np.arange(len(true_positives)) + 1)
        recall = cum_true_positives / num_ground_truths

        # Calculate the AP for the current class
        ap = average_precision_score(true_positives, scores) if len(scores) > 0 else 0
        class_aps.append(ap)

    mAP = np.mean(class_aps)  # Calculate the mean AP across all classes
    return mAP

# Train, Evaluation

In [ ]:
# Optimizer and scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
# Training + Validation Loop
num_epochs = 5

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    # Training Phase
    model.train()
    total_train_loss = 0

    for images, targets in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_train_loss += losses.item()

        # Backward pass and optimization
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Train Loss: {avg_train_loss:.4f}")

    # Validation Phase
    model.eval()
    all_predictions = []
    all_ground_truths = []

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Validation"):
            images = [img.to(device) for img in images]
            predictions = model(images)

            # save all predictions and ground truth results
            all_predictions.extend(predictions)
            all_ground_truths.extend(targets)

    # mAP
    mAP = evaluate_model(all_predictions, all_ground_truths, classes)
    print(f"Validation mAP: {mAP:.4f}\n")

# Test Prediction Visualization

In [ ]:
def visualize_prediction(image, prediction, classes, threshold=0.5):
    """
    Visualizes the prediction results on a single image with bounding boxes and class labels.

    Args:
        image (torch.Tensor): The input image tensor (C, H, W).
        prediction (dict): The model's prediction result, including 'boxes', 'labels', and 'scores'.
        classes (list): A list of class names corresponding to the numerical labels.
        threshold (float): The minimum confidence score required to visualize a prediction.
    """
    # Convert the image tensor from (C, H, W) to (H, W, C) format for visualization
    image = image.permute(1, 2, 0).numpy()

    # Create a matplotlib figure and axes for displaying the image
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image)  # Display the image

    # Visualize each bounding box and its corresponding label
    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        if score > threshold:  # Only visualize predictions above the confidence threshold
            x_min, y_min, x_max, y_max = box.tolist()  # Extract bounding box coordinates
            width, height = x_max - x_min, y_max - y_min  # Calculate the width and height of the box

            # Add the bounding box rectangle to the image
            rect = patches.Rectangle(
                (x_min, y_min), width, height, linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)

            # Add the class name and confidence score text above the bounding box
            ax.text(
                x_min,
                y_min - 10,
                f"{classes[label]}: {score:.2f}",  # Format the text: "Class Name: Confidence Score"
                color="red",
                fontsize=10,
                bbox=dict(facecolor="white", alpha=0.7),  # Add a white background for readability
            )

    plt.axis("off")  # Hide the axes for a cleaner visualization
    plt.show()  # Display the image with bounding boxes and labels

In [ ]:
model.eval()  # Set the model to evaluation mode
max_visualizations = 5  # Set the maximum number of visualizations
count = 0  # Initialize a counter for the number of visualizations
threshold=0.75 # Confidence Threshold

# No gradient calculation during validation (improves efficiency)
with torch.no_grad():
    # Iterate through the test data loader
    for i, (images, image_files) in enumerate(test_loader):
        # Move images to the appropriate device (GPU or CPU)
        images = [img.to(device) for img in images]
        # Make predictions with the model
        predictions = model(images)

        # Visualize each image and its predictions
        for img, pred, file_name in zip(images, predictions, image_files):
            # Exit the loop if the maximum number of visualizations has been reached
            if count >= max_visualizations:
                break  # Break out of the inner loop
            visualize_prediction(img.cpu(), pred, classes, threshold=threshold)
            count += 1  # Increment the visualization counter
        # Exit the loop if the maximum number of visualizations has been reached
        if count >= max_visualizations:
            break  # Break out of the inner loop

# SSD modified

This model leverages the compound scaling method proposed in the paper EfficientNet by Mingxing Tan and Quoc V. Le. The depth, width, and resolution are interrelated, and a balanced adjustment of these three factors is essential for stable model training. Additionally, this practice aims to enhance stability through 1x1 convolution (for dimensional expansion and reduction) while facilitating faster convergence during training via batch normalization (especially, compound scaling method will make the model little deeper). However, since this model is not pretrained, it may exhibit weaker generalization and slower convergence speed. To achieve satisfactory performance on the dataset, a higher number of epochs is likely required. Despite the application of strong augmentation, the model may still be vulnerable to overfitting due to the dataset size. Further regularization techniques such as dropout or weight decay could be explored to mitigate this issue.

- computational cost increased about 3 times much.
- rebuild the model in model simple structure and apply weight initialization instead
- pretrained weight applied

In [ ]:
# import torch.nn as nn
# import torch
# from collections import OrderedDict

# class SSDBBoneSK(nn.Module):
#     def __init__(self):
#         super().__init__()

#         def conv_bn_relu(in_channels, out_channels, kernel_size, stride=1, padding=0):
#             return nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
#                 nn.BatchNorm2d(out_channels),
#                 nn.ReLU(inplace=True)
#             )

#         self.feature1 = nn.Sequential(
#             conv_bn_relu(3, 2, kernel_size=1),
#             conv_bn_relu(2, 512, kernel_size=7, stride=2, padding=3),
#             conv_bn_relu(512, 256, kernel_size=1),
#             conv_bn_relu(256, 384, kernel_size=3, stride=2, padding=1),
#             conv_bn_relu(384, 256, kernel_size=1),
#             conv_bn_relu(256, 192, kernel_size=3, stride=2, padding=1)
#         )

#         self.feature2 = nn.Sequential(
#             conv_bn_relu(192, 128, kernel_size=1),
#             conv_bn_relu(128, 512, kernel_size=3, stride=1, padding=1),
#             nn.Dropout(p=0.3)
#         )

#         self.feature3 = nn.Sequential(
#             conv_bn_relu(512, 256, kernel_size=1),
#             conv_bn_relu(256, 384, kernel_size=3, stride=2, padding=1),
#             nn.Dropout(p=0.4)
#         )

#         self.feature4 = nn.Sequential(
#             conv_bn_relu(384, 256, kernel_size=1),
#             conv_bn_relu(256, 384, kernel_size=3, stride=2, padding=1),
#             nn.Dropout(p=0.5)
#         )

#         self.feature5 = nn.Sequential(
#             conv_bn_relu(384, 128, kernel_size=1),
#             conv_bn_relu(128, 256, kernel_size=3),
#         )

#         self.feature6 = nn.Sequential(
#             conv_bn_relu(256, 128, kernel_size=1),
#             conv_bn_relu(128, 256, kernel_size=3),
#         )

#         self.feature7 = nn.Sequential(
#             conv_bn_relu(256, 128, kernel_size=1),
#             conv_bn_relu(128, 256, kernel_size=4)
#         )

#         self._initialize_weights()

#     def _initialize_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#             elif isinstance(m, nn.BatchNorm2d):
#                 nn.init.constant_(m.weight, 1)
#                 nn.init.constant_(m.bias, 0)

#     def forward(self, x):
#         x1 = self.feature1(x)
#         x2 = self.feature2(x1)
#         x3 = self.feature3(x2)
#         x4 = self.feature4(x3)
#         x5 = self.feature5(x4)
#         x6 = self.feature6(x5)
#         x7 = self.feature7(x5)

#         output = [x1, x2, x3, x4, x5, x6, x7]
#         return OrderedDict([(str(i), v) for i, v in enumerate(output)])

In [ ]:
# def conv_bn_relu(in_channels, out_channels, kernel_size, stride=1, padding=0):
#     return nn.Sequential(
#         nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
#         nn.BatchNorm2d(out_channels),
#         nn.ReLU(inplace=True)
#     )

# class SSDBBoneSK(nn.Module):
#     def __init__(self, phi=1, alpha=1.2, beta=1.1, gamma=1.15):
#         super().__init__()
#         self.phi = phi
#         d = int(alpha ** phi)  # depth adjustment
#         w = int(beta ** phi)   # width adjustment

#         def make_layer(in_channels, mid_channels, out_channels, stride=1):
#             layers = []
#             for _ in range(d):
#                 layers.extend([
#                     conv_bn_relu(in_channels, mid_channels, kernel_size=1),  # 1x1 Conv
#                     conv_bn_relu(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1),  # 3x3 Conv
#                     conv_bn_relu(mid_channels, out_channels, kernel_size=1)  # 1x1 Conv
#                 ])
#                 in_channels = out_channels
#             return nn.Sequential(*layers)

#         self.feature1 = make_layer(3, 64, 512, stride=2)
#         self.feature2 = make_layer(512, 128, 512)
#         self.feature3 = make_layer(512, 256, 384, stride=2)
#         self.feature4 = make_layer(384, 256, 384, stride=2)
#         self.feature5 = make_layer(384, 128, 256)
#         self.feature6 = make_layer(256, 64, 128)
#         self.feature7 = make_layer(128, 32, 64)

#     def forward(self, x):
#         outputs = []
#         for i in range(1, 8):
#             x = getattr(self, f'feature{i}')(x)
#             outputs.append(x)
#         return OrderedDict([(str(i), v) for i, v in enumerate(outputs)])

import torch.nn as nn
import torch
from collections import OrderedDict
from torchvision import models

class SSDBBoneSK(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.pretrained_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        self.pretrained_layers = nn.Sequential(
            self.pretrained_model.conv1,
            self.pretrained_model.bn1,
            self.pretrained_model.relu,
            self.pretrained_model.maxpool,
            self.pretrained_model.layer1,
            self.pretrained_model.layer2,
            self.pretrained_model.layer3
        )

        def conv_bn_relu(in_channels, out_channels, kernel_size, stride=1, padding=0):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )

        self.feature1 = nn.Sequential(
            conv_bn_relu(1024, 512, kernel_size=1),
            conv_bn_relu(512, 256, kernel_size=3, stride=1, padding=1),
            nn.Dropout(p=0.3)
        )

        self.feature2 = nn.Sequential(
            conv_bn_relu(256, 128, kernel_size=1),
            conv_bn_relu(128, 256, kernel_size=3, stride=2, padding=1),
            nn.Dropout(p=0.4)
        )

        self.feature3 = nn.Sequential(
            conv_bn_relu(256, 128, kernel_size=1),
            conv_bn_relu(128, 256, kernel_size=3, stride=2, padding=1),
            nn.Dropout(p=0.5)
        )

        self.feature4 = nn.Sequential(
            conv_bn_relu(256, 64, kernel_size=1),
            conv_bn_relu(64, 128, kernel_size=3)
        )

        self.feature5 = nn.Sequential(
            conv_bn_relu(128, 64, kernel_size=1),
            conv_bn_relu(64, 128, kernel_size=3)
        )

        self.feature6 = nn.Sequential(
            conv_bn_relu(128, 64, kernel_size=1),
            conv_bn_relu(64, 128, kernel_size=4)
        )

        # self._initialize_weights()

    # def _initialize_weights(self):
    #     for m in self.modules():
    #         if isinstance(m, nn.Conv2d):
    #             nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    #         elif isinstance(m, nn.BatchNorm2d):
    #             nn.init.constant_(m.weight, 1)
    #             nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x1 = self.pretrained_layers(x)
        x2 = self.feature1(x1)
        x3 = self.feature2(x2)
        x4 = self.feature3(x3)
        x5 = self.feature4(x4)
        x6 = self.feature5(x5)
        x7 = self.feature6(x5)

        output = [x1, x2, x3, x4, x5, x6, x7]
        return OrderedDict([(str(i), v) for i, v in enumerate(output)])

In [ ]:
# model.apply(_initialize_weights)

# def _initialize_weights(self):
#     for m in self.modules():
#         if isinstance(m, nn.Conv2d):
#             nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#         elif isinstance(m, nn.BatchNorm2d):
#             nn.init.constant_(m.weight, 1)
#             nn.init.constant_(m.bias, 0)

In [ ]:
backbone = SSDBBoneSK()

# DefaultBoxGenerator: Method for generating the Anchor box
anchor_generator = DefaultBoxGenerator(
    aspect_ratios=[
        [0.5, 1, 2],        # 32x32
        [0.33, 0.5, 1, 2],  # 16x16
        [0.33, 0.5, 1, 2],  # 8x8
        [0.33, 0.5, 1, 2],  # 6x6
        [0.5, 1, 2],        # 4x4
        [0.5, 1, 2],        # 3x3
        [0.5, 1, 2],        # 3x3
    ],
    scales=[0.1, 0.2, 0.37, 0.54, 0.71, 0.88, 1.05, 1.2],
    steps=[8, 16, 32, 64, 100, 300, 512]
    )


model2 = ssd.SSD(
    backbone=backbone,
    anchor_generator=anchor_generator,
    size=(512, 512),
    num_classes=len(classes)  # Background, Cat, Dog
).to(device)

In [ ]:
params = [p for p in model2.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr=0.001,
    max_lr=0.01,
    step_size_up=800,
    mode='exp_range'
)

# optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
# lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Data Preprocessing

Because this model is not pretrained, strong augmentation may be necessary to achieve adequate generalization and prevent overfitting. Previously, I used conditional augmentation to mitigate class imbalance. Now, I will apply strong augmentation generally and incorporate class weighting in a subsequent step.

In [ ]:
from torchvision import transforms, tv_tensors

class VOCDataset(Dataset):
    def __init__(self, image_dir, annotation_dir, classes, image_list, df_input=None, transforms=None):
        self.image_dir = image_dir
        self.annotation_dir = annotation_dir
        self.classes = classes
        self.transforms = transforms
        self.image_files = image_list
        self.df_input = df_input

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx] + ".jpg"
        annotation_file = self.image_files[idx] + ".xml"
        image_path = os.path.join(self.image_dir, image_file)
        annotation_path = os.path.join(self.annotation_dir, annotation_file)

        image = Image.open(image_path).convert("RGB")
        image = np.array(image)

        boxes = []
        labels = []
        tree = ET.parse(annotation_path)
        root = tree.getroot()

        for obj in root.findall("object"):
            class_name = obj.find("name").text
            if class_name not in self.classes:
                continue
            labels.append(self.classes.index(class_name))
            bndbox = obj.find("bndbox")
            x_min = int(bndbox.find("xmin").text)
            y_min = int(bndbox.find("ymin").text)
            x_max = int(bndbox.find("xmax").text)
            y_max = int(bndbox.find("ymax").text)
            boxes.append([x_min, y_min, x_max, y_max])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = np.array(labels, dtype=np.int64)
        height, width, _ = image.shape


        # tensor to tv_tensor/ manual normalization(0 to 1)
        image = torch.tensor(image.transpose(2, 0, 1), dtype=torch.float32) / 255.0
        image = tv_tensors.Image(image)
        boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(height, width))

        # The key name must match the expected input format of the tv_tensor transform.
        # Transforms in torchvision v0.15+ expect dictionary keys like "image" and "bbox" to correctly apply transformations.
        if self.transforms:
            sample = {"image": image, "bbox": boxes}
            sample = self.transforms(sample)
            image = sample["image"]
            boxes = sample["bbox"]

        # Normalization applied to bounding boxes
        # boxes[..., 0] /= width
        # boxes[..., 1] /= height
        # boxes[..., 2] /= width
        # boxes[..., 3] /= height

        # boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(1, 1))
        target = {"boxes": boxes, "labels": torch.tensor(labels, dtype=torch.int64)}

        return image, target

In [ ]:
class TestDataset(Dataset):
    def __init__(self, image_dir, image_list, transforms=None):
        self.image_dir = image_dir
        self.image_files = image_list
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx] + ".jpg"
        image_path = os.path.join(self.image_dir, image_file)

        image = Image.open(image_path).convert("RGB")
        image = np.array(image)
        # tensor to tv_tensor/ manual normalization(0 to 1)
        image = torch.tensor(image.transpose(2, 0, 1), dtype=torch.float32) / 255.0

        if self.transforms:
            image = self.transforms(image)

        return image, self.image_files[idx]

## Augmentations for scratch model

In [ ]:
transforms = {
    "train": v2.Compose([
        v2.RandomHorizontalFlip(p=0.2),
        v2.RandomVerticalFlip(p=0.2),
        v2.RandomRotation(degrees=(-20, 20)),
        v2.ColorJitter(0.2, 0.2, 0.2, 0.2),
        # v2.ToImage(),
        v2.ToDtype(torch.float32, scale=False),
    ]),

    "val": v2.Compose([
        # v2.ToImage(),
        v2.ToDtype(torch.float32, scale=False),
    ]),

    "test": v2.Compose([
        # v2.ToImage(),
        v2.ToDtype(torch.float32, scale=False),
    ])
}

## Dataset declare and DataLoader

In [ ]:
# Create the training dataset
train_dataset = VOCDataset(
    image_dir=image_dir,
    annotation_dir=xml_dir,
    classes=classes,
    image_list=train_list,  # train_list
    df_input=df_trainval,
    transforms=transforms["train"]
)

# Create the validation dataset
val_dataset = VOCDataset(
    image_dir=image_dir,
    annotation_dir=xml_dir,
    classes=classes,
    image_list=val_list,  # val_list
    transforms=transforms["val"]
)

# Create the test dataset
test_dataset = TestDataset(
    image_dir=image_dir,
    image_list=test_taglist,  # test_list
    transforms=transforms["test"]
)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Display the sizes of the datasets
print(f"Train Dataset Size: {len(train_dataset)}")
print(f"Validation Dataset Size: {len(val_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")

In [ ]:
image, label = next(iter(train_loader))
print(label)

In [ ]:
image, label = next(iter(val_loader))
print(label)

In [ ]:
image[0].shape

In [ ]:
def denormalize_boxes(boxes, height, width):
    boxes[..., 0] *= width    # x_min
    boxes[..., 2] *= width    # x_max
    boxes[..., 1] *= height   # y_min
    boxes[..., 3] *= height   # y_max
    return boxes

## Visualization for augmented Image

In [ ]:
count = 0
for image, label in train_loader:
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))
    count += 1

    for i in range(8):
        ex_img = image[i]
        ex_label = label[i]

        ex_img_np = ex_img.permute(1,2,0).numpy()

        _, height, width = ex_img.shape

        # bounding box has been normalized during the preprocessing level
        # denormalize required in order to visualize
        # ex_label['boxes'] = denormalize_boxes(ex_label['boxes'].clone(), height, width)

        x_min, y_min, x_max, y_max = ex_label['boxes'].squeeze()

        width, height = x_max - x_min, y_max - y_min  # Calculate the width and height of the box

        # Add the bounding box rectangle to the image
        rect = patches.Rectangle(
            (x_min, y_min), width, height, linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.flat[i].add_patch(rect)

        # Add the class name and confidence score text above the bounding box
        label_index = ex_label['labels'][0].item() if isinstance(ex_label['labels'][0], torch.Tensor) else ex_label['labels'][0]  # Assuming labels is a list or tensor
        if 0 <= label_index < len(classes):  # check for valid index
            # Add the class name and confidence score text above the bounding box
            ax.flat[i].text(
                x_min,
                y_min - 10,
                classes[label_index],  # Use the label index to get class name
                color="red",
                fontsize=10,
                bbox=dict(facecolor="white", alpha=0.7),  # Add a white background for readability
        )

        ax.flat[i].imshow(ex_img_np)
        ax.flat[i].set_title(f"Example Image: {i}")
        ax.flat[i].axis("off")
    plt.tight_layout()
    plt.show()

    if count >= 1:
        break

Augmentations are correctly applied, and bounding box locations are adjusted accordingly.

In [ ]:
count = 0
for image, label in val_loader:
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))
    count += 1

    for i in range(8):
        ex_img = image[i]
        ex_label = label[i]

        ex_img_np = ex_img.permute(1,2,0).numpy()

        _, height, width = ex_img.shape

        # bounding box has been normalized during the preprocessing level
        # denormalize required in order to visualize
        # ex_label['boxes'] = denormalize_boxes(ex_label['boxes'].clone(), height, width)

        x_min, y_min, x_max, y_max = ex_label['boxes'].squeeze()

        width, height = x_max - x_min, y_max - y_min  # Calculate the width and height of the box

        # Add the bounding box rectangle to the image
        rect = patches.Rectangle(
            (x_min, y_min), width, height, linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.flat[i].add_patch(rect)

        # Add the class name and confidence score text above the bounding box
        label_index = ex_label['labels'][0].item() if isinstance(ex_label['labels'][0], torch.Tensor) else ex_label['labels'][0]  # Assuming labels is a list or tensor
        if 0 <= label_index < len(classes):  # check for valid index
            # Add the class name and confidence score text above the bounding box
            ax.flat[i].text(
                x_min,
                y_min - 10,
                classes[label_index],  # Use the label index to get class name
                color="red",
                fontsize=10,
                bbox=dict(facecolor="white", alpha=0.7),  # Add a white background for readability
        )

        ax.flat[i].imshow(ex_img_np)
        ax.flat[i].set_title(f"Example Image: {i}")
        ax.flat[i].axis("off")
    plt.tight_layout()
    plt.show()

    if count >= 1:
        break

In [ ]:
count = 0
for image, _ in test_loader:
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))
    count += 1

    for i in range(8):
        ex_img = image[i]

        ex_img_np = ex_img.permute(1,2,0).numpy()

        ax.flat[i].imshow(ex_img_np)
        ax.flat[i].set_title(f"Example Image: {i}")
        ax.flat[i].axis("off")
    plt.tight_layout()
    plt.show()

    if count >= 1:
        break

# Class weighting (opted out)

In [ ]:
# num_classes = len(classes)

# # Define background label index
# background_label = classes.index("background")  # 0

# # all_label (exclude background label)
# all_labels_objects = []
# for images, targets in tqdm(train_loader, desc="Collecting Labels"):
#     for target in targets:
#         all_labels_objects.extend(target["labels"].tolist())

# # object class weight balancing
# unique_object_labels = np.unique(all_labels_objects)
# class_weights_objects = compute_class_weight(
#     class_weight='balanced',
#     classes=unique_object_labels,
#     y=all_labels_objects
# )

# # Add background weight
# class_weights = np.zeros(num_classes)
# class_weights[unique_object_labels] = class_weights_objects
# class_weights[background_label] = 0.1 # background weight to lowest

# # make it in tensor
# class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Function modification

In [ ]:
def calculate_iou2(box, boxes):
    """
    Calculate Intersection over Union (IoU) between a box and multiple boxes.

    Args:
        box (array): Single bounding box [x_min, y_min, x_max, y_max].
        boxes (array): Array of bounding boxes [[x_min, y_min, x_max, y_max], ...].

    Returns:
        array: IoU scores for each box in `boxes`.
    """
    x_min = np.maximum(box[0], boxes[:, 0])
    y_min = np.maximum(box[1], boxes[:, 1])
    x_max = np.minimum(box[2], boxes[:, 2])
    y_max = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, x_max - x_min) * np.maximum(0, y_max - y_min)
    box_area = (box[2] - box[0]) * (box[3] - box[1])
    boxes_area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    union = box_area + boxes_area - intersection

    iou = intersection / (union + 1e-6)  # Avoid division by zero
    return iou

def calculate_ap2(true_positives, scores, num_ground_truths):
    if len(true_positives) == 0:
        return 0.0

    sorted_indices = np.argsort(-np.array(scores))
    true_positives = np.array(true_positives)[sorted_indices]

    cum_true_positives = np.cumsum(true_positives)
    precision = cum_true_positives / (np.arange(len(true_positives)) + 1)
    recall = cum_true_positives / num_ground_truths

    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([1.0], precision, [0.0]))

    for i in range(len(precision) - 1, 0, -1):
        precision[i - 1] = max(precision[i - 1], precision[i])

    indices = np.where(recall[1:] != recall[:-1])[0]
    ap = np.sum((recall[indices + 1] - recall[indices]) * precision[indices + 1])

    return ap

def evaluate_model2(predictions, ground_truths, classes):
    class_aps = []

    for class_idx, class_name in enumerate(classes[1:], start=1):
        true_positives = []
        scores = []
        num_ground_truths = 0

        for pred, gt in zip(predictions, ground_truths):
            pred_boxes = pred["boxes"][pred["labels"] == class_idx].cpu().numpy() if len(pred["boxes"]) > 0 else []
            pred_scores = pred["scores"][pred["labels"] == class_idx].cpu().numpy() if len(pred["scores"]) > 0 else []
            gt_boxes = gt["boxes"][gt["labels"] == class_idx].cpu().numpy() if len(gt["boxes"]) > 0 else []

            num_ground_truths += len(gt_boxes)
            matched = np.zeros(len(gt_boxes), dtype=bool)

            for box, score in zip(pred_boxes, pred_scores):
                ious = calculate_iou2(box, gt_boxes)
                max_iou_idx = np.argmax(ious) if len(ious) > 0 else -1
                max_iou = ious[max_iou_idx] if max_iou_idx >= 0 else 0

                if max_iou >= 0.5 and not matched[max_iou_idx]:
                    true_positives.append(1)
                    matched[max_iou_idx] = True
                else:
                    true_positives.append(0)

                scores.append(score)

        if num_ground_truths == 0:
            class_aps.append(0)
            continue

        ap = calculate_ap2(true_positives, scores, num_ground_truths)
        class_aps.append(ap)

    mAP = np.mean(class_aps) if len(class_aps) > 0 else 0.0
    return mAP

In [ ]:
def denorm_visualize_prediction(image, prediction, classes, threshold=0.5):
    """
    Visualizes the prediction results on a single image with bounding boxes and class labels.
    Additionally, denormalizing the bounding box, in order to visualize the image with the bounding box.

    Args:
        image (torch.Tensor): The input image tensor (C, H, W).
        prediction (dict): The model's prediction result, including 'boxes', 'labels', and 'scores'.
        classes (list): A list of class names corresponding to the numerical labels.
        threshold (float): The minimum confidence score required to visualize a prediction.
    """
    image = image.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image)
    img_height, img_width, _ = image.shape

    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        if score > threshold:
            box = denormalize_boxes(box.clone(), img_height, img_width)
            x_min, y_min, x_max, y_max = box.tolist()
            box_width, box_height = x_max - x_min, y_max - y_min

            rect = patches.Rectangle(
                (x_min, y_min), box_width, box_height, linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x_min, y_min - 10,
                f"{classes[label]}: {score:.2f}",
                color="red" if score > 0.75 else "blue",
                fontsize=10,
                bbox=dict(facecolor="white", alpha=0.7),
            )

    plt.axis("off")
    plt.show()

# Train and Validation

In [ ]:
# Training + Validation Loop
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    # Training Phase
    model2.train()
    total_train_loss = 0

    for images, targets in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model2(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_train_loss += losses.item()

        # Backward pass and optimization
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Train Loss: {avg_train_loss:.4f}")

    # Validation Phase
    model2.eval()
    all_predictions = []
    all_ground_truths = []

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Validation"):
            images = [img.to(device) for img in images]
            predictions = model2(images)

            # save all predictions and ground truth results
            all_predictions.extend(predictions)
            all_ground_truths.extend(targets)

    # mAP
    mAP = evaluate_model(all_predictions, all_ground_truths, classes)
    print(f"Validation mAP: {mAP:.4f}\n")

Class weighting has been opted out due to overfitting issues, as the class distributions are consistent across the train, validation, and test datasets.

In [ ]:
all_predictions[0]

# Test Visualization

In [ ]:
model2.eval()  # Set the model2 to evaluation mode
max_visualizations = 10  # Set the maximum number of visualizations
count = 0  # Initialize a counter for the number of visualizations
threshold=0.75 # Confidence Threshold

# No gradient calculation during validation (improves efficiency)
with torch.no_grad():
    # Iterate through the test data loader
    for i, (images, image_files) in enumerate(test_loader):
        # Move images to the appropriate device (GPU or CPU)
        images = [img.to(device) for img in images]
        # Make predictions with the model2
        predictions = model2(images)

        # Visualize each image and its predictions
        for img, pred, file_name in zip(images, predictions, image_files):
            visualize_prediction(img.cpu(), pred, classes, threshold=threshold)
            count += 1  # Increment the visualization counter
            # Exit the loop if the maximum number of visualizations has been reached
            if count >= max_visualizations:
                break  # Break out of the inner loop
        if count >= max_visualizations:
            break  # Break out of the inner loop